## Visualizing high_level_cost

Interactive exploration of the resource estimates: results per molecule, the error accounting, the compute-zone scans and the trade-offs between lookup options.  The parameters come from `resource_count/settings.py` (molecules and options) and `resource_count/error_accounting.py` (noise levels, target precision); change them there.  The two λ-scan cells take a few minutes each.

In [ ]:
# re-import resource_count's modules whenever their files change, so the
# kernel never runs a stale version while the code is being edited
%load_ext autoreload
%autoreload 2

import sys
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# make resource_count importable regardless of how the kernel was launched:
# add the project root (this notebook's parent directory) to sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from resource_count import high_level_cost as hl
from resource_count import accounting as acc
from resource_count import layout
from resource_count import cost_functions as rep_code     # the repetition-code cost model
from resource_count import cost_functions_ldpc as ldpc_code   # the hybrid LDPC cost model

# the figures' chart palette (scripts/figure_style.py, shared with the paper
# scripts): matplotlib's default colour cycle takes its series in order
if str(project_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(project_root / 'scripts'))
import figure_style as fs
plt.rcParams['axes.prop_cycle'] = fs.PROP_CYCLE

# Results

## FeMoCo-54

In [ ]:
# the noise level every cell below uses: '5e-4' or '1e-3' (the keys of NOISE,
# Error Accounting).  It picks the chosen distances from SETTINGS_54/76 for
# the cost tables, the error accounting, the physical qubits and the lookup
# sections.  Change it here and run all
NOISE_LEVEL = '5e-4'

# FeMoCo-54: the system (FEMOCO_54), and per cost model its compute-zone
# budget, block-encoding options, the distances chosen at each noise level
# and the distances the error-accounting scan runs (SETTINGS_54; what each
# entry means: resource_count.settings).  The cost tables, the error
# accounting and the later sections read them at NOISE_LEVEL.
# the system and its settings: resource_count.settings, the one place they are
# set (the scripts in scripts/ read them too) -- change them there
from resource_count.settings import FEMOCO_54, SETTINGS_54


def distance_text(distances):
    '''the distances for a title or printout, e.g. "d_rep = 16, d_LDPC = 16"'''
    return ', '.join(f'{k} = {v}' for k, v in distances.items())


def cost_tables(system, m, settings, level):
    '''Print high_level_cost.report for one block encoding of `system` (a dict
    of N, B, R, C, b_coef, b_rot) with cost model `m`, using that model's
    compute-zone budget, block-encoding options and the distances chosen at
    noise level `level` from `settings`.  Everything is passed in, so no
    other cell's variables can change it.  Returns the BE_cost record.'''
    p, s = system, settings[m]
    distances = s['distances'][level]
    print(f"{s['name']}, p_Z = {level}: {distance_text(distances)}, compute-zone budget {s['budget']}, "
          f"option {s['option']}")
    print(f"system: {p}\n")
    be = hl.BE_cost(p['N'], p['B'], p['R'], p['C'], p['b_coef'], p['b_rot'], s['budget'], distances,
                    model=m, buffer_budget=layout.buffer_zone_size(), **s['option'])
    hl.report(be, model=m)
    return be

In [ ]:
# cost tables of one FeMoCo-54 block encoding: rep code
be_rep_54 = cost_tables(FEMOCO_54, rep_code, SETTINGS_54, NOISE_LEVEL)

In [ ]:
# cost tables of one FeMoCo-54 block encoding: LDPC
be_ldpc_54 = cost_tables(FEMOCO_54, ldpc_code, SETTINGS_54, NOISE_LEVEL)

## FeMoCo-76

In [ ]:
# FeMoCo-76: the system (FEMOCO_76) and its settings (SETTINGS_76), as for
# FeMoCo-54 above.  cost_tables is defined in the FeMoCo-54 cell above.
# the system and its settings: resource_count.settings, the one place they are
# set (the scripts in scripts/ read them too) -- change them there
from resource_count.settings import FEMOCO_76, SETTINGS_76


In [ ]:
# cost tables of one FeMoCo-76 block encoding: rep code
be_rep_76 = cost_tables(FEMOCO_76, rep_code, SETTINGS_76, NOISE_LEVEL)

In [ ]:
# cost tables of one FeMoCo-76 block encoding: LDPC
be_ldpc_76 = cost_tables(FEMOCO_76, ldpc_code, SETTINGS_76, NOISE_LEVEL)

# Error Accounting

In [ ]:
# the run's fixed parameters (EPSILON, ROUND_TIME), the two noise levels
# (NOISE: per level the phase-flip rate p_Z, the magic-state error per Toffoli
# and the i-state error per i-state), the factory routing and the error model
# functions below: all from
# resource_count.error_accounting, which scripts/error_accounting_latex.py
# uses too -- change them there
from resource_count.error_accounting import (
    EPSILON, ROUND_TIME, NOISE, FACTORY_ROUTING,
    layer_errors, code_errors, algo_error)
assert NOISE_LEVEL in NOISE, f"NOISE_LEVEL {NOISE_LEVEL!r} (settings, Results) is not one of {list(NOISE)}"


# error_accounting's keys of each code's error, as error_accounting below names them
CODE_ERROR_NAME = {'rep': 'Rep code error', 'ldpc': 'LDPC error'}


def total_cost(distances, option):
    '''Total_time for the current parameters at `distances` (the model's
    distance keywords, e.g. dict(d_rep=15)), with epsilon, the
    round time, the cost model and its compute-zone budget set in the parameter
    cell, and `option` the block-encoding options, dict(method=..., K=...,
    lam=..., outer_variety=..., inner_variety=...), passed in explicitly so
    no other cell's variables can change them.'''
    return hl.Total_time(N, B, R, C, b_coef, b_rot, ancilla_budget, buffer_budget, distances, be_const,
                          epsilon=epsilon, round_time=round_time,
                          model=model, **option)

def error_accounting(distances, p_Z, p_magic, p_i, option, factory_routing):
    '''The run time of one run, the failure probabilities, added (an upper
    bound), and the expected duration: the run time divided by the
    probability that a run succeeds.  When the failure probability reaches 1
    a run is not expected to succeed, so the duration is infinite (np.inf)
    rather than negative.  The algorithmic error is the codes' errors added
    (code_errors: 'Rep code error', its volume including the factory
    routing, and 'LDPC error' when the model has an LDPC layer), with the
    cost model set in the parameter cell, as total_cost uses.'''
    cost = total_cost(distances, option)
    magic_error = cost['total Toffoli']*p_magic
    i_error = cost['total i-state']*p_i
    codes = code_errors(cost, distances, p_Z, model, factory_routing)
    algo = algo_error(cost, distances, p_Z, model, factory_routing)
    fail = algo + magic_error + i_error
    time = cost['total days']/(1-fail) if fail < 1 else np.inf
    share = layer_errors(cost, distances, p_Z, model, factory_routing)['factory routing']['share']
    return {'Run time': cost['total days'], 'Algo error': algo,
            **{CODE_ERROR_NAME[code]: e for code, e in codes.items()},
            'Magic state error': magic_error, 'i-state error': i_error, 'Failure prob': fail,
            'Duration': time, 'Factory routing share of volume': share}


In [ ]:
# FeMoCo-54: the system and the chosen model's settings, read from FEMOCO_54
# and SETTINGS_54 (Results section above) -- change them there, not here
model = ldpc_code                  # cost model: rep_code or ldpc_code
N, B, R, C, b_coef, b_rot = (FEMOCO_54[k] for k in ('N', 'B', 'R', 'C', 'b_coef', 'b_rot'))
buffer_budget = layout.buffer_zone_size()
ancilla_budget = SETTINGS_54[model]['budget']     # compute-zone budget
option = SETTINGS_54[model]['option']             # block-encoding options
scan = SETTINGS_54[model]['scan']                 # the distances scanned
dist_accounting = SETTINGS_54[model]['distances'][NOISE_LEVEL]   # chosen at the noise level
epsilon, round_time = EPSILON, ROUND_TIME   # set with NOISE (Error Accounting)
be_const = FEMOCO_54['be_const']
noise = NOISE[NOISE_LEVEL]        # the noise level, set at the top (settings, Results)
p_Z, p_magic, p_i = noise['p_Z'], noise['p_magic'], noise['p_i']
print(f"FeMoCo-54, {SETTINGS_54[model]['name']}, p_Z = {NOISE_LEVEL}: error accounting at "
      f"{distance_text(dist_accounting)}\n")

for dist in scan:
    errors = error_accounting(dist, p_Z, p_magic, p_i, option=option, factory_routing=FACTORY_ROUTING[model])
    print(f"{distance_text(dist)}: run {errors['Run time']:.3f} days, failure {errors['Failure prob']:.3g} (logical {errors['Algo error']:.3g}, "
          f"magic {errors['Magic state error']:.3g}, i-state {errors['i-state error']:.3g}), "
          f"expected duration {errors['Duration']:.3f} days")

visual = layer_errors(total_cost(dist_accounting, option), dist_accounting, p_Z, model, FACTORY_ROUTING[model])
print(f"factory routing ({FACTORY_ROUTING[model]} physical qubits): {visual['factory routing']['share']:.2%} of the logical qubit-cycles")

In [ ]:
# FeMoCo-76: the system and the chosen model's settings, read from FEMOCO_76
# and SETTINGS_76 (Results section above) -- change them there, not here
model = ldpc_code                  # cost model: rep_code or ldpc_code
N, B, R, C, b_coef, b_rot = (FEMOCO_76[k] for k in ('N', 'B', 'R', 'C', 'b_coef', 'b_rot'))
buffer_budget = layout.buffer_zone_size()
ancilla_budget = SETTINGS_76[model]['budget']     # compute-zone budget
option = SETTINGS_76[model]['option']             # block-encoding options
scan = SETTINGS_76[model]['scan']                 # the distances scanned
dist_accounting = SETTINGS_76[model]['distances'][NOISE_LEVEL]   # chosen at the noise level
epsilon, round_time = EPSILON, ROUND_TIME   # set with NOISE (Error Accounting)
be_const = FEMOCO_76['be_const']
noise = NOISE[NOISE_LEVEL]        # the noise level, set at the top (settings, Results)
p_Z, p_magic, p_i = noise['p_Z'], noise['p_magic'], noise['p_i']
print(f"FeMoCo-76, {SETTINGS_76[model]['name']}, p_Z = {NOISE_LEVEL}: error accounting at "
      f"{distance_text(dist_accounting)}\n")

for dist in scan:
    errors = error_accounting(dist, p_Z, p_magic, p_i, option=option, factory_routing=FACTORY_ROUTING[model])
    print(f"{distance_text(dist)}: run {errors['Run time']:.3f} days, failure {errors['Failure prob']:.3g} (logical {errors['Algo error']:.3g}, "
          f"magic {errors['Magic state error']:.3g}, i-state {errors['i-state error']:.3g}), "
          f"expected duration {errors['Duration']:.3f} days")

x = error_accounting(dist_accounting, p_Z, p_magic, p_i, option=option, factory_routing=FACTORY_ROUTING[model])
print(f"factory routing ({FACTORY_ROUTING[model]} physical qubits): {x['Factory routing share of volume']:.2%} of the logical qubit-cycles")

## Distance choice at both noise levels

For each system, noise level and cost model: the expected duration (run time divided by the probability that a run succeeds) at every scanned distance, and the distance chosen. The shortest expected duration wins; a choice within `TIE` of it with smaller distances (fewer qubits) is preferred. `∞`: a run never succeeds (failure probability ≥ 1); `–`: the model refuses the distance (reason printed below the table).

In [ ]:
# Distance choice at both noise levels (NOISE), for both cost models.  The
# scan (DISTANCE_SCAN), the tie margin (TIE) and the rule (choose_distance)
# are resource_count.error_accounting's, which scripts/error_accounting_latex.py
# uses too.  Each run is costed once per distance (Total_time, with the
# model's compute-zone budget and options from SETTINGS_54 / SETTINGS_76; the
# cost does not depend on the noise) and its failure probability evaluated at
# each noise level: logical (algo_error, each layer at its own distance, the
# factory routing included) + magic-state + i-state errors, added.
# Everything is passed in, so no other cell's variables change it.
# Re-running this cell clears the cache
from resource_count import error_accounting as ea
from resource_count.error_accounting import DISTANCE_SCAN, TIE
_run_cache = {}


def run_cost(system, settings, m, dist):
    '''Total_time for `system` with model `m` at `dist` (cached per cell run)'''
    key = (id(system), m.__name__, tuple(sorted(dist.items())))
    if key not in _run_cache:
        s = settings[m]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')     # budget warnings are not the point here
            _run_cache[key] = ea.total_time(system, s['budget'], s['option'], dist, m, EPSILON, ROUND_TIME)
    return _run_cache[key]


def choose_distance(system, settings, m, noise):
    '''every scanned distance with its run time, failures and expected
    duration, and the chosen one (error_accounting.choose_distance).  A
    distance the model refuses (e.g. the LDPC angle storage check) is kept as
    infeasible, with the model's reason'''
    return ea.choose_distance(lambda dist: run_cost(system, settings, m, dist), DISTANCE_SCAN[m], noise, m,
                              FACTORY_ROUTING[m], TIE)


def show_distance_choice(name, system, settings):
    print(name)
    for level, noise in NOISE.items():
        print(f"  p_Z = {level}  (p_magic {noise['p_magic']:g}, p_i {noise['p_i']:g})")
        for m in (rep_code, ldpc_code):
            rows, c = choose_distance(system, settings, m, noise)
            print(f"    {settings[m]['name']}: {distance_text(c['dist'])}: run {c['days']:.3f} d, failure "
                  f"{c['fail']:.3f} (logical {c['logical']:.3f}, magic {c['magic']:.3f}, i-state "
                  f"{c['istate']:.3f}), expected {c['expected']:.3f} d; factory routing "
                  f"{c['factory']:.2%} of the logical qubit-cycles")
            d_reps = sorted({r['dist']['d_rep'] for r in rows})
            fmt = lambda r: ('     \u2013' if 'infeasible' in r else '     \u221e' if r['expected'] == np.inf
                             else f"{r['expected']:6.3f}") + ('*' if r is c else ' ')
            print('      expected (days), d_rep  ' + ''.join(f'{d:>7}' for d in d_reps))
            if m is rep_code:
                print('                              ' + ''.join(fmt(r) for r in rows))
            else:
                for dl in (12, 16, 22):
                    print(f'      d_LDPC = {dl:<2}              ' + ''.join(fmt(r) for r in rows if r['dist']['d_LDPC'] == dl))
            for reason in sorted({r['infeasible'] for r in rows if 'infeasible' in r}):
                print(f'      \u2013 infeasible: {reason}')
    print()


def check_settings_distances(name, system, settings):
    '''the distances typed into `settings` (SETTINGS_54 / SETTINGS_76) against
    this cell's choice, per model and noise level'''
    for level, noise in NOISE.items():
        for m in (rep_code, ldpc_code):
            _, c = choose_distance(system, settings, m, noise)
            typed = settings[m]['distances'][level]
            status = 'matches' if typed == c['dist'] else f"DIFFERS: the choice is {distance_text(c['dist'])}"
            print(f"  {name}, p_Z = {level}, {settings[m]['name']}: settings {distance_text(typed)} -- {status}")


show_distance_choice('FeMoCo-54', FEMOCO_54, SETTINGS_54)
show_distance_choice('FeMoCo-76', FEMOCO_76, SETTINGS_76)
print('The chosen distances in SETTINGS_54 / SETTINGS_76 against this choice:')
check_settings_distances('FeMoCo-54', FEMOCO_54, SETTINGS_54)
check_settings_distances('FeMoCo-76', FEMOCO_76, SETTINGS_76)

## Algorithmic error by code

For every molecule, cost model and noise level, at the chosen distances (`SETTINGS_54` / `SETTINGS_76`, independent of `NOISE_LEVEL`): the algorithmic error split into its codes, each $\varepsilon = 1 - (1 - p)^V$ at the code's own distance, the rep code's $V$ including the factory routing (`error_accounting.code_errors`), and the failure probability $p_\mathrm{fail} = \varepsilon_\mathrm{rep} + \varepsilon_\mathrm{LDPC} + \varepsilon_{C_XC_XX} + \varepsilon_i$.

In [ ]:
# the algorithmic error by code, at the settings' chosen distances, for every
# molecule, model and noise level (run_cost and the noise levels from the
# distance choice above)
def show_code_errors(molecules):
    print(f"{'':<11}{'model':<10}{'p_Z':<6}{'distances':<26}{'eps_rep':>10}{'eps_LDPC':>10}{'eps_alg':>10}"
          f"{'eps_CCX':>10}{'eps_i':>10}{'p_fail':>10}")
    for name, (system, settings) in molecules.items():
        for m in (rep_code, ldpc_code):
            for level, noise in NOISE.items():
                dist = settings[m]['distances'][level]
                cost = run_cost(system, settings, m, dist)
                codes = code_errors(cost, dist, noise['p_Z'], m, FACTORY_ROUTING[m])
                fail, logical, magic, istate = ea.failure(cost, dist, noise, m, FACTORY_ROUTING[m])
                assert logical == sum(codes.values())
                ldpc = f"{codes['ldpc']:10.6f}" if 'ldpc' in codes else f"{'--':>10}"
                print(f"{name:<11}{settings[m]['name']:<10}{level:<6}{distance_text(dist):<26}{codes['rep']:10.6f}"
                      f"{ldpc}{logical:10.6f}{magic:10.6f}{istate:10.6f}{fail:10.6f}")


from resource_count.settings import MOLECULES
show_code_errors(MOLECULES)

## Run volumes

For every molecule, cost model and noise level, at the chosen distances (`SETTINGS_54` / `SETTINGS_76`, independent of `NOISE_LEVEL`), the full algorithm (all block encodings): its duration in logical cycles (rounds / rounds per cycle: $d$ for the rep code, $\max(d_\mathrm{rep}, d_\mathrm{LDPC})$ for LDPC) and its effective volumes in logical qubit-cycles (`error_accounting.run_volumes`, as `scripts/volume_latex.py` tabulates them):

- algorithm: `Total_time`'s total effective volume, each layer converted at its own distance;
- factory routing (`FACTORY_ROUTING`: 1180 physical qubits in the rep code, 1020 in LDPC) and factories (`FACTORIES`: 2154 in the rep code, 1958 in LDPC, one $|i\rangle$ factory fewer): active for the whole run, converted like rep-layer volume, physical qubit-rounds / $d_\mathrm{rep}(2d_\mathrm{rep}-1)$;
- total: the three added;
- effective logical qubits per layer: the layer's volume divided by its own logical cycles (rounds / the layer's distance), i.e. the average number of logical qubits in it, the rep layer's including the factory routing and the factories; and their sum.

In [ ]:
# the full algorithm's duration (logical cycles) and effective volumes
# (logical qubit-cycles), at the settings' chosen distances, for every
# molecule, model and noise level (run_cost from the distance choice above)
from resource_count.error_accounting import FACTORIES


def show_run_volumes(molecules):
    print(f"{'':<11}{'model':<10}{'p_Z':<6}{'distances':<26}{'cycles':>11}{'algorithm':>11}"
          f"{'routing':>11}{'factories':>11}{'total':>11}{'qubits rep':>11}{'LDPC':>7}{'total':>7}")
    for name, (system, settings) in molecules.items():
        for m in (rep_code, ldpc_code):
            for level in NOISE:
                dist = settings[m]['distances'][level]
                v = ea.run_volumes(run_cost(system, settings, m, dist), dist, m, FACTORY_ROUTING[m], FACTORIES[m])
                print(f"{name:<11}{settings[m]['name']:<10}{level:<6}{distance_text(dist):<26}"
                      + ''.join(f"{v[k]:11.3e}" for k in ('cycles', 'algorithm', 'factory routing', 'factories', 'total'))
                      + f"{v['logical qubits [rep]']:11.0f}"
                      + (f"{v['logical qubits [ldpc]']:7.0f}" if 'logical qubits [ldpc]' in v else f"{'--':>7}")
                      + f"{v['logical qubits']:7.0f}")


show_run_volumes(MOLECULES)

## Rep code and LDPC with each other's lookup options

Each model with its own chosen options, and with the other's, all at the same distances (`cross_options.CROSS_DISTANCES`: $d_\mathrm{rep} = 15$ for the rep code, $d_\mathrm{rep} = 15$, $d_\mathrm{LDPC} = 16$ for LDPC), so the options are compared at nearly the same distance; the errors at `NOISE_LEVEL`. The variants and their runs are `resource_count.cross_options`' (a contained example; the rest of the package is unchanged), as in `scripts/cross_options_latex.py`:

- **LDPC, rep options**: the LDPC model running the rep code's algorithm, a plain QROM with $\lambda = 2$ in RPrep+Rot and clean SelectSwap in both preps, at LDPC's compute-zone budget.
- **rep, dirty options**: the rep model with LDPC's chosen options for the same molecule, i.e. its dirty lookups (dirty SelectSwap in RPrep+Rot: restricted, $K = 2$, for FeMoCo-54, full access, $K = 4$, for FeMoCo-76, $\lambda = 2$; dirty SelectSwap in InnerPrep; `SETTINGS_54` / `SETTINGS_76`). **Not possible in the rep-code layout as it is**: it needs Z access to the system qubits, i.e. access corridors under them, one per two system qubits ($N$ in all). It is costed with `cross_options.REP_WITH_SYSTEM_Z_ACCESS`, a copy of the rep model (`rep_code` itself is unchanged). The corridors are in its physical qubits (each half of the system zone $\lceil 3N/2\rceil + 2$ wide) and, taken as present for the whole run, in its volume and failure probability, an upper bound.

Then the full cost tables of the two cross variants for `COMPARE_DETAIL`.

In [ ]:
# Rep code and LDPC with each other's lookup options, all at the same
# distances (CROSS_DISTANCES: d_rep = 15 for the rep code, d_rep = 15,
# d_LDPC = 16 for LDPC), so the options are compared at nearly the same
# distance in both models; the errors at NOISE_LEVEL.  The variants and their
# runs are resource_count.cross_options' (a contained example, the rest of the
# package unchanged), as in scripts/cross_options_latex.py:
#   'LDPC, rep options'   the LDPC model running the rep code's algorithm
#                         (QROM, lam = 2; clean SelectSwap in both preps), at
#                         LDPC's compute-zone budget
#   'rep, dirty options'  the rep model with LDPC's chosen options for the
#                         same molecule, i.e. its dirty lookups (dirty
#                         SelectSwap in RPrep+Rot and InnerPrep).  NOT
#                         possible in the rep-code layout as it is: it needs Z
#                         access to the system qubits, i.e. access corridors
#                         under them, one per two system qubits (N in all).
#                         Costed with cross_options.REP_WITH_SYSTEM_Z_ACCESS
#                         (rep_code itself is untouched); the corridors are in
#                         its physical qubits (each system half ceil(3N/2) + 2
#                         wide) and, taken as present for the whole run, in its
#                         volume and p_fail (an upper bound)
from resource_count import cross_options as xo

COMPARE_DETAIL = 'FeMoCo-54'      # the molecule whose full cost tables are printed at the end


def show_cross_options(molecules, level):
    noise = NOISE[level]
    print(f"p_Z = {level}: the whole run, at {distance_text(xo.CROSS_DISTANCES[rep_code])} (rep code) and "
          f"{distance_text(xo.CROSS_DISTANCES[ldpc_code])} (LDPC)\n")
    print(f"{'':<11}{'variant':<20}{'corridors':>10}{'physical':>10}{'days':>8}{'Toffolis':>11}{'i-states':>11}"
          f"{'p_fail':>9}{'expected':>10}")
    for name, (system, settings) in molecules.items():
        for v in xo.cross_variants(system, settings):
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                r = xo.cross_run(system, v, noise, FACTORY_ROUTING[xo.chip_model(v)], EPSILON, ROUND_TIME)
            phys = xo.physical_qubits(system, v)
            cost = r['cost']
            print(f"{name:<11}{v['label']:<20}{v['corridors']:>10}{phys:>10,}{cost['total days']:8.3f}"
                  f"{cost['total Toffoli']:11.3e}{cost['total i-state']:11.3e}{r['fail']:9.4f}{r['expected']:10.3f}")
        print()
    print("'corridors': the access corridors added under the system qubits (logical slots); only the dirty rep "
          "variant needs them.  They are in its physical qubits and, present for the whole run, in its p_fail")


def show_cross_detail(molecules, name):
    '''the full cost tables (high_level_cost.report) of one block encoding of
    the two cross variants, for molecule `name`'''
    system, settings = molecules[name]
    for v in xo.cross_variants(system, settings)[1::2]:
        print(f"===== {name}, {v['label']}: {distance_text(v['distances'])}, compute-zone budget {v['budget']}, "
              f"option {v['option']}")
        if v['corridors']:
            print(f"(these per-block-encoding tables exclude the {v['corridors']} access corridors; they cost no rounds)")
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            be = hl.BE_cost(system['N'], system['B'], system['R'], system['C'], system['b_coef'], system['b_rot'],
                            v['budget'], v['distances'], model=v['model'], buffer_budget=layout.buffer_zone_size(),
                            **v['option'])
        hl.report(be, model=v['model'])
        print()


show_cross_options(MOLECULES, NOISE_LEVEL)
show_cross_detail(MOLECULES, COMPARE_DETAIL)


## Rep code with $\lambda = 3$

The rep code with RPrep+Rot loading $\lambda = 3$ rotation angles per lookup instead of the chosen $\lambda = 2$ (`SETTINGS_54`), everything else as chosen, for FeMoCo-54; the errors at `NOISE_LEVEL`. It runs at the distances chosen at `NOISE_LEVEL` for $\lambda = 2$, and at its own: the distance choice (`error_accounting.choose_distance` over `DISTANCE_SCAN` with `TIE`, as in Distance choice and Growing the compute zone) for the $\lambda = 3$ option at its budget, $d_\mathrm{rep} = 13$ at $p_Z = 5\times10^{-4}$.

$\lambda = 3$ needs a compute zone of $72$: $48$ clean ancilla and the $24$ access corridors of the output register (`batching.batching_cost`), so it runs at a compute-zone budget of `LAM3_BUDGET` $= 72$; the cell checks that each option fits its budget. Its physical qubits include the wider compute zone, $w_\text{comp} = \lceil 3\lambda b_\text{rot}/2\rceil$ (Physical qubits, below).

Then the full cost tables of one $\lambda = 3$ block encoding, at its own distances.

In [ ]:
# The rep code with RPrep+Rot loading lam = 3 angles per lookup instead of the
# chosen lam = 2 (SETTINGS_54), everything else as chosen, for FeMoCo-54; the
# errors at NOISE_LEVEL.  lam = 3 runs at the distances chosen at NOISE_LEVEL
# for lam = 2, and at its own: the distance choice (ea.choose_distance over
# DISTANCE_SCAN with TIE, as the distance choice above and lambda_scan) for
# the lam = 3 option at its budget.  lam = 3 needs a compute zone of 72 (48
# clean ancilla + 24 access corridors of the output register,
# batching.batching_cost), so its budget is LAM3_BUDGET; each option is
# checked to fit its budget.  The physical qubits include the wider compute
# zone, w_comp = ceil(3 lam b_rot / 2)
import warnings

from resource_count import batching as bt
from resource_count import error_accounting as ea
from resource_count.error_accounting import total_time, failure, physical_qubits

LAM3_BUDGET = 72      # the compute zone lam = 3 needs in the rep code


def lam3_distances(system, settings, level):
    '''the lam = 3 option's own distances at noise level `level`: the
    distance choice (ea.choose_distance) over the rep code's DISTANCE_SCAN'''
    option = dict(settings[rep_code]['option'], lam=3)

    def run(distances):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', UserWarning)
            return total_time(system, LAM3_BUDGET, option, distances, rep_code, EPSILON, ROUND_TIME)
    _, chosen = ea.choose_distance(run, ea.DISTANCE_SCAN[rep_code], NOISE[level], rep_code, FACTORY_ROUTING[rep_code],
                                   ea.TIE)
    return chosen['dist']


def lam3_variants(system, settings, level):
    '''the rep code's chosen option at its budget and distances, the same
    with lam = 3 at LAM3_BUDGET at those distances, and at its own
    (lam3_distances): (label, budget, option, distances)'''
    s = settings[rep_code]
    lam3 = dict(s['option'], lam=3)
    return [('chosen, lam = 2', s['budget'], s['option'], s['distances'][level]),
            ('lam = 3', LAM3_BUDGET, lam3, s['distances'][level]),
            ('lam = 3, own d', LAM3_BUDGET, lam3, lam3_distances(system, settings, level))]


def show_lam3(name, system, settings, level):
    noise = NOISE[level]
    print(f"{name}, rep code, p_Z = {level}\n")
    print(f"{'option':<17}{'d_rep':>6}{'budget':>7}{'zone':>6}{'physical':>10}{'days':>8}{'Toffolis':>11}"
          f"{'i-states':>11}{'eps_alg':>9}{'eps_CXCXX':>10}{'eps_i':>8}{'p_fail':>8}{'expected':>10}")
    for label, budget, option, distances in lam3_variants(system, settings, level):
        zone = bt.batching_cost(system['N'], system['B'], system['R'], system['b_rot'], distances,
                                option['method'], option['lam'], option['K'], model=rep_code)['Compute zone']
        assert zone <= budget, (label, zone, budget)
        cost = total_time(system, budget, option, distances, rep_code, EPSILON, ROUND_TIME)
        fail, logical, magic, istate = failure(cost, distances, noise, rep_code, FACTORY_ROUTING[rep_code])
        expected = cost['total days']/(1 - fail) if fail < 1 else np.inf
        phys = physical_qubits(system, rep_code, option, distances)['total']
        print(f"{label:<17}{distances['d_rep']:>6}{budget:>7}{zone:>6}{phys:>10,}{cost['total days']:8.3f}"
              f"{cost['total Toffoli']:11.3e}{cost['total i-state']:11.3e}{logical:9.5f}{magic:10.5f}"
              f"{istate:8.5f}{fail:8.4f}{expected:10.3f}")
    print("\n'zone': the compute zone the option needs (clean ancilla + access corridors), at most its budget;"
          "\n'own d': the lam = 3 option's own distance choice; days and expected without slack")


def show_lam3_detail(name, system, settings, level):
    '''the full cost tables (high_level_cost.report) of one lam = 3 block
    encoding, at its own distances'''
    label, budget, option, distances = lam3_variants(system, settings, level)[2]
    print(f"===== {name}, rep code, {label}: {distance_text(distances)}, compute-zone budget {budget}, "
          f"option {option}")
    be = hl.BE_cost(system['N'], system['B'], system['R'], system['C'], system['b_coef'], system['b_rot'],
                    budget, distances, model=rep_code, buffer_budget=layout.buffer_zone_size(), **option)
    hl.report(be, model=rep_code)


show_lam3('FeMoCo-54', FEMOCO_54, SETTINGS_54, NOISE_LEVEL)
print()
show_lam3_detail('FeMoCo-54', FEMOCO_54, SETTINGS_54, NOISE_LEVEL)


## Growing the compute zone

The fastest block encoding at each compute-zone size, for both cost models and both molecules, at `NOISE_LEVEL` (set at the top of the notebook, Results; `resource_count.lambda_scan`, which says in detail what is chosen and how). RPrep+Rot loads the $N-1$ rotation angles in $J = \lceil (N-1)/\lambda\rceil$ batches, so only $\lambda$ values that change $J$ matter: each level is the smallest $\lambda$ for its $J$, from $\lambda = 2$ (the settings' budget) to $J = 1$. Its compute zone is $\lambda b_\text{rot}$ qubits and their access corridors (none in LDPC).

At each level: every RPrep+Rot option viable in that zone (batch sizes $1..\lambda$, $K$ up to $2^6$, which reaches every viable $K$) with every prep variety the model allows, ranked by expected duration at the settings' distances; the 5 fastest are then scanned over all distances (`error_accounting.choose_distance`) and the shortest expected duration wins. Physical qubits are those of the chip with the level's compute zone. The durations carry the paper tables' $5\%$ slack (`SLACK`, `scripts/error_accounting_latex.py`), so they compare to them; a common factor, it does not change the choice.

The scan takes a few minutes; the plot is in the cell after it, so it can be restyled without rerunning the scan. `python scripts/lambda_scan.py <level>` prints the same table for all four cases.

In [ ]:
# the fastest block encoding at each compute-zone size (lambda_scan.scan),
# per molecule and cost model, at NOISE_LEVEL: a table per case.  Shares are
# of one block encoding's rounds, each subroutine with its adjoint
from resource_count import lambda_scan
from resource_count.settings import MOLECULES

# the paper tables' slack on the durations, from scripts/error_accounting_latex.py
if str(project_root / 'scripts') not in sys.path:
    sys.path.insert(0, str(project_root / 'scripts'))
from error_accounting_latex import SLACK

LAMBDA_ALPHA_MAX = 6      # RPrep+Rot's swap parameters K up to 2**6: every viable K (lambda_scan)
LAMBDA_TOP = 5            # stage-1 candidates rescanned over the distances
LAMBDA_MODELS = {'rep code': rep_code, 'LDPC': ldpc_code}


def lambda_option_text(o):
    lookup = 'qrom' if o['method'] == 'qrom' else f"{o['method']} K={o['K']}"
    return f"{lookup} lam={o['lam']}, preps {o['outer_variety'][0]}/{o['inner_variety'][0]}"


lambda_scans = {}         # (molecule, code) -> lambda_scan.scan rows
for mol, (system, settings) in MOLECULES.items():
    for code, m in LAMBDA_MODELS.items():
        rows = lambda_scan.scan(system, m, settings, NOISE_LEVEL, lam_max=system['N'] - 1,
                                alpha_max=LAMBDA_ALPHA_MAX, top=LAMBDA_TOP, slack=SLACK)
        lambda_scans[(mol, code)] = rows
        print(f"\n{mol}, {code}, p_Z = {NOISE_LEVEL}")
        print(f"{'lam':>4}{'J':>4}{'zone':>6}  {'option':<36}{'dist':>7}{'days':>8}{'p_fail':>8}"
              f"{'<t>':>8}{'physical':>10}{'RPrep+Rot':>10}{'InnerPrep':>10}{'OuterPrep':>10}")
        for r in rows:
            sh = r['shares']
            print(f"{r['lam']:>4}{r['J']:>4}{r['budget']:>6}  {lambda_option_text(r['option']):<36}"
                  f"{'/'.join(str(v) for v in r['distances'].values()):>7}{r['days']:8.3f}{r['fail']:8.4f}"
                  f"{r['expected']:8.3f}{r['phys']:>10,}{100*sh['RPrep+Rot']:9.1f}%"
                  f"{100*sh['InnerPrep']:9.1f}%{100*sh['OuterPrep']:9.1f}%")
print(f"\npreps: outer/inner variety (c clean, d dirty); days, <t>: with {100*(SLACK - 1):.0f}% slack")

In [ ]:
# the scan above against lambda: expected duration, physical qubits, failure
# probability and the round shares of RPrep+Rot and InnerPrep, per molecule
# (columns), both cost models in each panel.  The levels are evenly spaced,
# labelled by lambda and the number of batches J
# the codes as the chart palette's series (figure_style: the rep code S1
# yellow, LDPC S2 teal), as in the paper's figures; RPrep+Rot filled, InnerPrep open
from matplotlib.lines import Line2D

LAMBDA_CODE_SERIES = {'rep code': 0, 'LDPC': 1}
LAMBDA_CODE_MARKER = {'rep code': 'o', 'LDPC': 's'}
LAMBDA_INK = fs.INK
PANELS = [('expected', f'expected duration (days, {100*(SLACK - 1):.0f}% slack)'), ('phys', 'physical qubits (thousands)'),
          ('fail', 'failure probability'), ('shares', 'share of the block encoding\'s rounds (%)')]

mols = list(MOLECULES)
fig, axes = plt.subplots(len(PANELS), len(mols), figsize=(11, 12), sharex='col')
for col, mol in enumerate(mols):
    levels = [(r['lam'], r['J']) for r in lambda_scans[(mol, 'rep code')]]
    x = range(len(levels))
    for row, (field, label) in enumerate(PANELS):
        ax = axes[row, col]
        for code, series in LAMBDA_CODE_SERIES.items():
            rows = lambda_scans[(mol, code)]
            assert [(r['lam'], r['J']) for r in rows] == levels
            if field == 'shares':
                for sub, filled in (('RPrep+Rot', True), ('InnerPrep', False)):
                    fs.plot(ax, x, [100*r['shares'][sub] for r in rows], series,
                            marker=LAMBDA_CODE_MARKER[code], lw=1.6, ms=4.5, filled=filled)
            else:
                scale = 1e-3 if field == 'phys' else 1
                fs.plot(ax, x, [scale*r[field] for r in rows], series, marker=LAMBDA_CODE_MARKER[code],
                        lw=1.6, ms=4.5)
        if field == 'shares':
            ax.axhline(50, color=LAMBDA_INK['secondary'], lw=0.8, ls=':')
            ax.set_ylim(0, 110)
            ax.set_yticks(range(0, 101, 20))
        else:
            ax.set_ylim(bottom=0)
        ax.set_xticks(list(x))
        ax.set_xticklabels([f'{l}\n{J}' for l, J in levels], fontsize=8)
        ax.grid(color=LAMBDA_INK['grid'], lw=0.6)
        ax.set_axisbelow(True)
        for side in ('top', 'right'):
            ax.spines[side].set_visible(False)
        if col == 0:
            ax.set_ylabel(label, color=LAMBDA_INK['secondary'])
        if row == 0:
            ax.set_title(f'{mol}, p_Z = {NOISE_LEVEL}', loc='left', color=LAMBDA_INK['primary'])
        if row == len(PANELS) - 1:
            ax.set_xlabel('λ (angles per batch) / J (batches)', color=LAMBDA_INK['secondary'])
        if col == len(mols) - 1 and row == 0:
            ax.legend([fs.handle(s, marker=LAMBDA_CODE_MARKER[c], lw=1.6, ms=4.5)
                       for c, s in LAMBDA_CODE_SERIES.items()], list(LAMBDA_CODE_SERIES),
                      frameon=False, fontsize=8)
        if col == len(mols) - 1 and field == 'shares':      # above the lines, which stay below 85%
            ax.legend([Line2D([], [], ls='none', marker='o', ms=4.5, mec=fs.OUTLINE,
                              mfc=fs.OUTLINE if filled else 'white') for filled in (True, False)],
                      ['RPrep+Rot', 'InnerPrep'], frameon=False, fontsize=8, loc='upper center', ncol=2)
fig.suptitle('The fastest block encoding per compute-zone size (λ b_rot and its access corridors)',
             x=0.01, ha='left', color=LAMBDA_INK['primary'])
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

In [ ]:
# the same scan as the paper's trade-off figure: expected duration against
# physical qubits, one point per compute-zone size, a panel per molecule, at
# NOISE_LEVEL -- scripts/lambda_scan.py's tradeoff_figure (what it marks and
# how it labels: its docstring), so the notebook and the paper's PDF are the
# same figure.  LAMBDA_SHOW_LOW26 overlays Low+26's frontier
# (Low et al., A Denser Planar Surface Code, arXiv:2605.30455, their
# Table XXVII: a surface-code estimate under its own noise model).  Their
# data is not included: see lambda_scan_script.LOW26_MISSING.  Shown inside the script's FIG_STYLE,
# as the script saves it
import lambda_scan as lambda_scan_script     # scripts/lambda_scan.py, not resource_count.lambda_scan

LAMBDA_SHOW_LOW26 = False
LAMBDA_SCRIPT_CODES = {'rep code': 'rep', 'LDPC': 'ldpc'}    # this notebook's code names -> the script's

fig = lambda_scan_script.tradeoff_figure(
    {f'{mol}|{LAMBDA_SCRIPT_CODES[code]}': rows for (mol, code), rows in lambda_scans.items()},
    NOISE_LEVEL, low=LAMBDA_SHOW_LOW26)
with plt.rc_context(lambda_scan_script.FIG_STYLE):
    plt.show()

In [ ]:
# the numbers of the paper's lambda-sweep paragraph, from the scan above
# (lambda_scans at NOISE_LEVEL, durations with SLACK): each curve's first
# step, InnerPrep's share at lambda = 2 and at J = 1, the smallest space-time
# volume (lambda_scan.space_time_volume: physical qubits x expected
# duration) and the footprint from which the repetition-cat architecture is
# at least as fast as LDPC-cat (lambda_scan.footprint_crossover: each
# architecture's best expected duration within a qubit budget)
print(f'p_Z = {NOISE_LEVEL}')

print('\nthe first step, lambda = 2 -> the next level')
for mol in MOLECULES:
    for code in LAMBDA_MODELS:
        a, b = lambda_scans[(mol, code)][:2]
        print(f"  {mol}, {code}: lam {a['lam']} -> {b['lam']}: {a['phys']:,} -> {b['phys']:,} physical qubits, "
              f"<t> {a['expected']:.3f} -> {b['expected']:.3f} days, p_fail {100*a['fail']:.2f}% -> "
              f"{100*b['fail']:.2f}%, {distance_text(a['distances'])} -> {distance_text(b['distances'])}")

print("\nInnerPrep's share of the block encoding's rounds (with its adjoint)")
inner = {(mol, code): [100*r['shares']['InnerPrep'] for r in lambda_scans[(mol, code)]]
         for mol in MOLECULES for code in LAMBDA_MODELS}
for (mol, code), s in inner.items():
    print(f"  {mol}, {code}: lam = 2 {s[0]:.1f}%, J = 1 {s[-1]:.1f}%")
print(f"  range: lam = 2 {min(s[0] for s in inner.values()):.1f}-{max(s[0] for s in inner.values()):.1f}%, "
      f"J = 1 {min(s[-1] for s in inner.values()):.1f}-{max(s[-1] for s in inner.values()):.1f}%")

print('\nthe smallest space-time volume (physical qubits x expected duration)')
for mol in MOLECULES:
    for code in LAMBDA_MODELS:
        r = min(lambda_scans[(mol, code)], key=lambda_scan.space_time_volume)
        print(f"  {mol}, {code}: {lambda_scan.space_time_volume(r)/1e6:.3f} megaqubit-hours at lam = {r['lam']} "
              f"(J = {r['J']}): {r['phys']:,} physical qubits, <t> {r['expected']:.3f} days")

print('\nthe faster architecture at a footprint: best <t> within each qubit budget')
for mol in MOLECULES:
    rep, ldpc = lambda_scans[(mol, 'rep code')], lambda_scans[(mol, 'LDPC')]
    last, first = lambda_scan.footprint_crossover(rep, ldpc)
    print(f"  {mol}: LDPC-cat faster at budgets up to {last:,} physical qubits, repetition-cat at least "
          f"as fast from {first:,}")
    for q in sorted({r['phys'] for r in rep + ldpc}):
        if 0.8*last <= q <= first:
            print(f"    {q:>7,} qubits: LDPC {lambda_scan.best_within(ldpc, q):.3f} days, "
                  f"rep code {lambda_scan.best_within(rep, q):.3f} days")

In [ ]:
# the footprints at the two noise levels: each curve's physical qubits at
# FOOTPRINT_OTHER against those at the reference FOOTPRINT_REFERENCE, at the
# same lambda (the levels depend on N only), as a percentage of the
# reference, with the mean over each curve's levels and over all curves'
# levels.  Reuses lambda_scans for NOISE_LEVEL and scans the other level
# (lambda_scan.scan, as above: a few minutes)
FOOTPRINT_REFERENCE, FOOTPRINT_OTHER = '5e-4', '1e-3'

footprint_scans = {}      # level -> (molecule, code) -> lambda_scan.scan rows
for level in (FOOTPRINT_REFERENCE, FOOTPRINT_OTHER):
    if level == NOISE_LEVEL:
        footprint_scans[level] = lambda_scans
        continue
    footprint_scans[level] = {
        (mol, code): lambda_scan.scan(system, m, settings, level, lam_max=system['N'] - 1,
                                      alpha_max=LAMBDA_ALPHA_MAX, top=LAMBDA_TOP, slack=SLACK)
        for mol, (system, settings) in MOLECULES.items() for code, m in LAMBDA_MODELS.items()}

print(f'physical qubits at p_Z = {FOOTPRINT_OTHER} against p_Z = {FOOTPRINT_REFERENCE}, at the same lambda')
footprint_all = []
for mol in MOLECULES:
    for code in LAMBDA_MODELS:
        ref, other = (footprint_scans[level][(mol, code)] for level in (FOOTPRINT_REFERENCE, FOOTPRINT_OTHER))
        assert [r['lam'] for r in ref] == [r['lam'] for r in other], (mol, code)
        diff = [100*(b['phys'] - a['phys'])/a['phys'] for a, b in zip(ref, other)]
        footprint_all += diff
        print(f"\n  {mol}, {code}: mean +{sum(diff)/len(diff):.1f}% over {len(diff)} levels "
              f"(+{min(diff):.1f} to +{max(diff):.1f}%)")
        for a, b, d in zip(ref, other, diff):
            print(f"    lam {a['lam']:>3}: {a['phys']:>7,} -> {b['phys']:>7,} (+{d:.1f}%), "
                  f"{distance_text(a['distances'])} -> {distance_text(b['distances'])}")
print(f'\nmean over all curves: +{sum(footprint_all)/len(footprint_all):.1f}% ({len(footprint_all)} levels)')


In [ ]:
# the qubits Low+26's surface-code estimate needs to be as fast as each
# point of our curves: for each scan row (lambda level) at runtime t, our
# expected duration with SLACK, Q_surf(t) is the fewest qubits of a Low+26
# point whose expected runtime is at most t (lambda_scan.fewest_as_fast),
# and the matched-runtime ratio is Q_surf(t)/Q_cat(t), our physical qubits.
# Low+26's points: every row of their Table XXVII for the molecule, the
# frontier and the highlighted rows below it, which are not all on the
# frontier -- with them the match is never worse for Low+26.  Their table
# is not included in this repository: see lambda_scan_script.LOW26_MISSING.  Their expected runtime under their own noise model, ours
# at each of footprint_scans' levels (the cell above)
low26_frontier = lambda_scan_script.load_low26()

if low26_frontier is None:
    print(lambda_scan_script.LOW26_MISSING)
else:
    LOW26_POINTS = {mol: [(1e3*q, expected/24) for q, _, expected, _, _ in
                          low26_frontier.FRONTIER[mol] + low26_frontier.SELECTED[mol]]
                    for mol in MOLECULES}       # (physical qubits, expected days)

    for level, scans in footprint_scans.items():
        print(f'p_Z = {level}: Low+26 qubits / ours at the same runtime (Low+26 at most as long as ours)')
        for mol in MOLECULES:
            low = LOW26_POINTS[mol]
            print(f"\n  {mol}: Low+26 expected runtimes {24*min(t for _, t in low):.2f} h to "
                  f"{max(t for _, t in low):.1f} d")
            for code in LAMBDA_MODELS:
                ratios = []
                for r in scans[(mol, code)]:
                    match = lambda_scan.fewest_as_fast(low, r['expected'])
                    assert match is not None, (level, mol, code, r['lam'], r['expected'])   # inside their range
                    ratios.append((match[0]/r['phys'], r, match))
                ref = ratios[0]
                lo, hi = min(ratios, key=lambda x: x[0]), max(ratios, key=lambda x: x[0])
                print(f"    {code}: {lo[0]:.1f}-{hi[0]:.1f}x over {len(ratios)} levels; "
                      f"lam = {ref[1]['lam']} (reference) {ref[0]:.1f}x")
                for ratio, r, (q, t) in ratios:
                    print(f"      lam {r['lam']:>3}: ours {r['phys']:>7,} qubits, {r['expected']:.3f} d ({24*r['expected']:5.1f} h)"
                          f" | Low+26 {q/1e3:7.1f}k qubits, {24*t:6.2f} h | {ratio:5.1f}x")
        print()


## Physical qubits

The chip's physical qubit count for each cost model, from its global layout.  The algorithmic core is counted without its ancilla qubits and then doubled; the factory zone is set per model (`FACTORY_ROUTING`, `FACTORIES`, as in the error accounting).  The rep code has one $|C_XC_XX\rangle$ factory (1566 qubits) and three $|i\rangle$ factories ($d_C = 7$, mid-long protocol; $3\cdot 196 = 588$ qubits), $2154$ in all, with routing $2(22\cdot 10 + 29\cdot 12 + 2\cdot 11) = 1180$ qubits.  LDPC needs one $|i\rangle$ factory fewer, $1566 + 2\cdot 196 = 1958$ qubits, so its routing has 10 rows instead of 12 under the $|C_XC_XX\rangle$ factory and no $2\cdot 11$: $2(22\cdot 10 + 29\cdot 10) = 1020$ qubits.  Both are the same at every distance.

**Repetition code.**  Two stacked rectangles, each one row of logical qubits with routing rows above or below:

$$\text{Space}^\text{rep}_\text{algo} = (w_\text{comp} + b_\text{rot} + n_X + n_{B+1} + 3)(d+5) + (N+2)(d+4) + (N+2)d + 27d$$

$$\text{Space}^\text{rep}_\text{tot} = 2\,\text{Space}^\text{rep}_\text{algo} + \underbrace{1180}_\text{factory routing} + \underbrace{2154}_\text{factories}$$

- $d = d_\text{rep}$.
- Top rectangle, one routing row above and four below ($d+5$): the compute zone, $w_\text{comp} = \lceil 3\lambda b_\text{rot}/2 \rceil$ with $\lambda$ the rep code's RPrep+Rot batch size; the Fourier resource state, $b_\text{rot}$; the address registers $n_X + n_{B+1}$ (as `BE_cost` sets them: $n_X = \lceil\log_2 X\rceil$, $X = N + 2^{\lceil\log_2 R\rceil}C$, $n_{B+1} = \lceil\log_2(B+1)\rceil$), the two spin-selection qubits and the flag qubit ($+3$).
- The system qubits in two halves of $N+2$ ($N$ system qubits, a QPE qubit and an access corridor): the top half with one routing row above and three below ($d+4$), the bottom half with none ($d$).
- The buffer zone, $27 = $ `layout.buffer_zone_size()`, set by Add.

**Hybrid LDPC.**  An LDPC layer holding the algorithmic qubits, on a flip chip with a repetition-code layer underneath for routing, the buffer and the resource states:

$$\text{Space}^\text{LDPC}_\text{tot} = \underbrace{2\big((H-1)L + \Delta_H\big)}_\text{LDPC layer} + 2\big(\underbrace{HL + \Delta_H + 2L + 3(\ell_\text{comp} + 22) + 27d}_\text{repetition-code layer}\big) + \underbrace{1020}_\text{factory routing} + \underbrace{1958}_\text{factories}$$

$$L = \ell_\text{address} + \ell_\text{comp} + N + 3$$

- The LDPC layer: $H$ rows of width $L$, plus $\Delta_H$ for the $Z$ logical operators at the sides, from `ldpc_code.chip_layout(d_LDPC)` (`H` and `base`).  It holds $HL + \Delta_H$ data and $(H-2)L + \Delta_H$ ancilla qubits, $2\big((H-1)L + \Delta_H\big)$ in all, and up to $2L$ logical qubits, two per column (their $X$ logical operators in the bottom two rows).
- Along $L$: the address registers, $\ell_\text{address} = \lceil (n_X + n_{B+1})/2 \rceil$; the compute zone, $\ell_\text{comp}$; the system qubits, $N$ ($2N$ qubits, two per column); and $3$ for the three spin-selection and flag qubits and the two QPE qubits, stacked at the right edge.
- The compute zone holds the phase gradient (Fourier resource) state and the compute qubits, two logical qubits per column.  It is filled in blocks of three columns, two Fourier qubits stacked and then a $2\times 2$ square of compute qubits, until all $b_\text{rot}$ Fourier qubits are placed: $\lceil 3b_\text{rot}/2 \rceil$ columns, the minimal width that lays out the phase gradient addition at no extra space.  Of its slots the first $b_\text{rot}$ hold the phase gradient state and the rest are compute qubits; compute qubits beyond those are added to the right:
  $$\ell_\text{comp} = \max\big(\lceil 3b_\text{rot}/2 \rceil,\ \lceil (b_\text{rot} + n_\text{comp})/2 \rceil\big), \qquad n_\text{comp} = n_\text{angles}\, b_\text{rot},$$
  with $n_\text{angles}$ the rotation angles RPrep+Rot's lookup holds in the compute zone: $K\lambda$ for the clean SelectSwap (all $K$ copies), $\lambda$ for the QROM and the dirty SelectSwaps (checked against `batching.batching_cost`'s compute zone; the LDPC layer has no access corridors).
- The repetition-code layer: the routing layer under the LDPC layer, $HL + \Delta_H$; two routing rows along the bottom, $2L$; three routing rows under the compute zone up to the buffer's right edge, $3(\ell_\text{comp} + 22)$: the buffer sits 5 columns into the compute zone, so the routing runs $27 - 5 = 22$ columns beyond it; and the buffer, $27d$ with $d = d_\text{rep}$.  $H$ and $\Delta_H$ are set by $d_\text{LDPC}$.

Printed below: every term at the distances in `SETTINGS_54` / `SETTINGS_76`, then the totals at the distances the distance choice above picks, at both noise levels.


In [ ]:
# Physical qubits of the chip per cost model (formulas in the markdown above),
# for a system (FEMOCO_54 / FEMOCO_76), RPrep+Rot's lookup option and the
# distances, all passed in: resource_count.error_accounting's, which
# scripts/error_accounting_latex.py uses too.  The factory zone is set
# per model
from resource_count.error_accounting import FACTORIES, physical_qubits


PHYSICAL_SYSTEMS = (('FeMoCo-54', FEMOCO_54, SETTINGS_54), ('FeMoCo-76', FEMOCO_76, SETTINGS_76))


def check_compute_qubits(system, settings, level):
    """LDPC: the compute qubits must be what batching counts as RPrep+Rot's
    compute zone (no access corridors in the LDPC layer), at the distances
    chosen at noise level `level`"""
    from resource_count import batching as bt
    s = settings[ldpc_code]
    distances = s['distances'][level]
    p = physical_qubits(system, ldpc_code, s['option'], distances)
    row = bt.batching_cost(system['N'], system['B'], system['R'], system['b_rot'], distances,
                           s['option']['method'], s['option']['lam'], s['option']['K'], model=ldpc_code)
    assert row['Compute zone'] == p['params']['n_comp'], (row['Compute zone'], p['params'])


def show_physical_terms(name, system, settings, level):
    """every term at the distances chosen at noise level `level`"""
    for m in (rep_code, ldpc_code):
        s = settings[m]
        distances = s['distances'][level]
        p = physical_qubits(system, m, s['option'], distances)
        print(f"{name}, {s['name']}, p_Z = {level}: {distance_text(distances)}, option {s['option']}")
        print('  ' + ', '.join(f'{k} = {v}' for k, v in p['params'].items())
              + f"  (compute-zone budget in the settings: {s['budget']})")
        for label, n in p['terms']:
            print(f'    {label:<62}{n:>8,}')
        print(f"    {'total':<62}{p['total']:>8,}")
    print()


def show_physical_at_chosen_distances(systems):
    """the totals at the distances the distance choice picks (choose_distance), at both noise levels"""
    print(f"{'':<12}{'p_Z':<7}{'model':<10}{'distances':<26}{'physical qubits':>16}{'expected days':>15}")
    for name, system, settings in systems:
        for level, noise in NOISE.items():
            for m in (rep_code, ldpc_code):
                _, c = choose_distance(system, settings, m, noise)
                p = physical_qubits(system, m, settings[m]['option'], c['dist'])
                print(f"{name:<12}{level:<7}{settings[m]['name']:<10}{distance_text(c['dist']):<26}"
                      f"{p['total']:>16,}{c['expected']:>15.3f}")


# (the loops leave the notebook's own names, e.g. system, untouched)
for entry in PHYSICAL_SYSTEMS:
    check_compute_qubits(*entry[1:], NOISE_LEVEL)
for entry in PHYSICAL_SYSTEMS:
    show_physical_terms(*entry, NOISE_LEVEL)
show_physical_at_chosen_distances(PHYSICAL_SYSTEMS)


## RPrep+Rot lookup options and trade-offs

Every lookup table batching option (method, K, λ) for both codes, searched by `optimize.search_rprep_rot` and `batching.batching_table`:

- **Trade-off**: each viable option costed as a whole RPrep+Rot, Toffoli cost against logical cycles. The line joins the trade-off front (options no other option beats on both); the ring marks the one with the fewest cycles.
- **Option map**: every option the search sweeps, with its compute zone (clean ancilla + access corridors). Grey cells are not viable, with the first reason why: no Z access to the system qubits (dirty methods, rep code), swap source too small, adder layout (λ = 1), or over budget.
- **Table**: the trade-off figure's numbers.

Each system parameter set has its own compute-zone budget per code: the rep code's includes the access corridors of the output register, the LDPC model needs none (32). Set `system` to `'A'` or `'B'`.

In [ ]:
from resource_count import batching as bt
from resource_count import optimize as opt
from resource_count import cost_functions_ldpc as ldpc_code   # the hybrid LDPC cost model

# the two system parameter sets, each with its compute-zone budget per code
# (from FEMOCO_54 / FEMOCO_76 and SETTINGS_54 / SETTINGS_76): the rep code's
# includes the output register's access corridors, LDPC needs none
SYSTEMS = {key: dict(**{k: system[k] for k in ('N', 'B', 'R', 'C', 'b_rot')},
                     budget={'rep code': settings[rep_code]['budget'], 'LDPC': settings[ldpc_code]['budget']})
           for key, (system, settings) in (('A', (FEMOCO_54, SETTINGS_54)), ('B', (FEMOCO_76, SETTINGS_76)))}
system = 'A'                      # which parameter set to show

# each parameter set's settings (Results section), for its options and the
# distances chosen at each noise level
SETTINGS_BY_SYSTEM = {'A': SETTINGS_54, 'B': SETTINGS_76}


def code_distances(sys_key, m, level):
    '''the distances chosen for parameter set `sys_key` and cost model `m` at
    noise level `level`, from SETTINGS_54 / SETTINGS_76'''
    return SETTINGS_BY_SYSTEM[sys_key][m]['distances'][level]


# the cost model per code, and its distances: those chosen for the parameter
# set shown (`system`) at NOISE_LEVEL (settings, Results), a dict of the
# model's own distance keywords passed to every routine
CODES = {
    'rep code': dict(model=rep_code, distances=code_distances(system, rep_code, NOISE_LEVEL)),
    'LDPC': dict(model=ldpc_code, distances=code_distances(system, ldpc_code, NOISE_LEVEL)),
}
lams, alpha_max = range(1, 5), 3  # search range: lam = 1..4, K up to 2**3

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

# chart ink (figure_style.INK) on white, and the lookup methods as the chart
# palette's series S1-S4 (figure_style), each with its own marker too, so
# colour never carries identity alone.  'edge' is the marker edge: S1's
# yellow keeps a near-black one
INK = dict(fs.INK, surface='#ffffff')
VIABLE, REJECTED = fs.area(0), fs.area(5)     # option map: S1 yellow, S6 grey hatched
METHOD_STYLE = {
    'qrom':             dict(color=fs.S1, edge=fs.OUTLINE, marker='o', label='qrom (K = 1)'),
    'clean':            dict(color=fs.S2, edge=fs.S2, marker='D', label='clean SelectSwap'),
    'dirty_restricted': dict(color=fs.S3, edge=fs.S3, marker='^', label='dirty, restricted access'),
    'dirty_full':       dict(color=fs.S4, edge=fs.S4, marker='s', label='dirty, full access'),
}
SHORT = {'qrom': 'qrom', 'clean': 'clean', 'dirty_restricted': 'dirty', 'dirty_full': 'dirty'}
REASON = {'no system Z access': 'no Z access', 'swap source': 'swap source',
          'adder layout': 'adder layout', 'compute zone': 'over budget'}
thousands = FuncFormatter(lambda v, _: f'{v:,.0f}')


def style_axes(ax):
    '''recessive chrome: hairline solid grid, no top/right spines'''
    ax.set_facecolor(INK['surface'])
    ax.grid(color=INK['grid'], linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    for side in ('left', 'bottom'):
        ax.spines[side].set_color(INK['axis'])
    ax.tick_params(colors=INK['muted'], labelcolor=INK['secondary'])


def search(code):
    '''optimize.search_rprep_rot for `code` on the chosen system'''
    p, c = SYSTEMS[system], CODES[code]
    return opt.search_rprep_rot(p['N'], p['B'], p['R'], p['C'], p['b_rot'], c['distances'],
                                lams=lams, alpha_max=alpha_max,
                                ancilla_budget=p['budget'][code], model=c['model'])


def option_label(r):
    K = '' if r['method'] == 'qrom' else f"K={r['K']}, "
    return f"{SHORT[r['method']]} {K}λ={r['size of batches']}"

In [ ]:
# RPrep+Rot options (optimize.search_rprep_rot): Toffoli count against logical
# cycles -- scripts/rprep_rot_tradeoff.py's tradeoff_figure, so the notebook
# and the paper's PDF (`pixi run figures`, figs/rprep_rot_tradeoff.pdf) are
# the same figure.  The script draws FeMoCo-54 at p_Z = 5e-4; this cell draws
# the parameter set (`system`) and NOISE_LEVEL chosen above.  Shown inside the
# script's PRINT, as the script saves it
import rprep_rot_tradeoff as rprep_rot_script   # scripts/rprep_rot_tradeoff.py

distance_tex = rprep_rot_script.distance_tex    # the cells below label distances with it

results = {code: search(code) for code in CODES}
fig = rprep_rot_script.tradeoff_figure(results, CODES)
with plt.rc_context(rprep_rot_script.PRINT):
    plt.show()

In [ ]:
# option map: every (method, K, lam) the search sweeps, with its compute zone
# (clean ancilla + access corridors); grey cells name the first reason the
# option is not viable (bt.show(..., include_rejected=True) lists them all)
ROWS = [('qrom', 1)] + [(m, 2**a) for m in ('clean', 'dirty_restricted', 'dirty_full')
                        for a in range(1, alpha_max + 1)]
ROW_LABELS = ['qrom'] + [f"{m.replace('_', ' ')}  K={K}" for m, K in ROWS[1:]]
gap = 0.04

fig, axes = plt.subplots(1, len(CODES), figsize=(12, 0.42*len(ROWS) + 1.8), sharey=True)
fig.patch.set_facecolor(INK['surface'])
for ax, code in zip(axes, CODES):
    p, c = SYSTEMS[system], CODES[code]
    rows = bt.batching_table(p['N'], p['B'], p['R'], p['b_rot'], c['distances'], lams=lams,
                             alpha_max=alpha_max, clean_budget=p['budget'][code],
                             include_rejected=True, model=c['model'])
    cell = {(r['method'], r['K'], r['lam']): r for r in rows}
    ax.set_facecolor(INK['surface'])
    for i, (m, K_row) in enumerate(ROWS):
        for j, lam_col in enumerate(lams):
            r = cell[(m, K_row, lam_col)]
            ok = r['viable']
            ax.add_patch(Rectangle((j + gap, i + gap), 1 - 2*gap, 1 - 2*gap,
                                   **(VIABLE if ok else REJECTED)))
            ax.text(j + 0.5, i + (0.5 if ok else 0.36), r['Compute zone'], ha='center',
                    va='center', fontsize=10, color=INK['primary'] if ok else INK['secondary'],
                    fontweight='bold' if ok else 'normal')
            if not ok:
                ax.text(j + 0.5, i + 0.7, REASON[r['rejected'][0]], ha='center', va='center',
                        fontsize=7.5, color=INK['secondary'])
    ax.set_xlim(0, len(lams))
    ax.set_ylim(len(ROWS), 0)
    ax.set_xticks([j + 0.5 for j in range(len(lams))], [f'λ = {lam}' for lam in lams])
    ax.set_yticks([i + 0.5 for i in range(len(ROWS))], ROW_LABELS)
    ax.tick_params(length=0, labelcolor=INK['secondary'])
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_title(f"{code}  ·  {distance_text(c['distances'])}  ·  compute zone ≤ {p['budget'][code]}",
                 color=INK['primary'], fontsize=11, loc='left')

fig.legend(handles=[Rectangle((0, 0), 1, 1, **VIABLE, label='viable'),
                    Rectangle((0, 0), 1, 1, **REJECTED,
                              label='not viable (first reason shown)')],
           loc='lower center', ncol=2, frameon=False, fontsize=9,
           labelcolor=INK['secondary'], bbox_to_anchor=(0.5, -0.02))
fig.suptitle(f'Lookup options, parameter set {system}: compute zone '
             '(clean ancilla + access corridors) per method, K and λ',
             color=INK['primary'], fontsize=12, x=0.01, ha='left')
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

In [ ]:
# every option the search sweeps, viable or not, costed as a whole RPrep+Rot
# (optimize.search_rprep_rot(..., include_rejected=True)): the viable ones first
# (* trade-off front, <- chosen), then the rejected ones with their reasons.
# clean = the lookup's clean ancilla, corr = its access corridors, zone = both
for code in CODES:
    p, c = SYSTEMS[system], CODES[code]
    print(f"{code}, {distance_text(c['distances'])}, compute zone <= {p['budget'][code]}: whole RPrep+Rot")
    opt.show_options(opt.search_rprep_rot(p['N'], p['B'], p['R'], p['C'], p['b_rot'], c['distances'],
                                          lams=lams, alpha_max=alpha_max,
                                          ancilla_budget=p['budget'][code], model=c['model'],
                                          include_rejected=True),
                     model=c['model'], distances=c['distances'])
    print()

In [ ]:
# the lookup table alone (batching.py), every option viable or not: lookups,
# lookup Toffoli and rounds, clean ancilla, access corridors, compute zone, the
# corridors a dirty method would need under the system qubits, and the reasons
# an option is rejected.  Not comparable across lam (the per-batch work is left out)
for code in CODES:
    p, c = SYSTEMS[system], CODES[code]
    print(f"{code}, {distance_text(c['distances'])}, compute zone <= {p['budget'][code]}: lookup table only")
    bt.show(bt.batching_table(p['N'], p['B'], p['R'], p['b_rot'], c['distances'], lams=lams,
                              alpha_max=alpha_max, clean_budget=p['budget'][code],
                              include_rejected=True, model=c['model']))
    print()

## SelectSwap against a plain QROM: one lookup

**Why the Toffoli count can mislead in LDPC.** A clean SelectSwap trades Select (QROM) Toffolis for Swap Toffolis. Both codes count the same Toffolis, but not the same rounds per Toffoli:

- **rep code**: a Toffoli step takes max(d, 21) rounds in both the QROM and the Swap (21 is the C_XC_XX time, so it is d only from d = 21). The QROM additionally spends d rounds per address on its parity CNOTs, which the Swap applies at the same time as its Toffolis: about d + ½·max(d, 21) rounds per QROM Toffoli (25.5 at d = 15) against max(d, 21) per Swap Toffoli (21). So rounds fall in step with the Toffoli count, or slightly faster;
- **LDPC**: a Swap Toffoli takes 2d rounds (32 at d = 16), more than a QROM Toffoli (about 26.5), so rounds fall less than the Toffoli count suggests.

**What is plotted.** For each table size (`lookup_addresses`), each code and each K = 1, 2, 4, 8, 16: one clean SelectSwap lookup, `model.SelectSwap_cost(addresses, lookup_bits, log2 K, d, 'clean')`, on its own: the Select (a skew-tree QROM over ⌈addresses/K⌉ entries, writing K copies of the output), the Swap network (moving the selected copy into place), and measuring out the K − 1 copies not selected. K = 1 is a plain QROM over all the addresses. Nothing else is included: no phase fixup, no idle qubits, no budget.

- **y**: the lookup's Toffoli count, or its rounds, divided by the same code's plain QROM over the same addresses (K = 1 is 100 %; lower is cheaper).
- **dotted black line**: the Toffoli count. It is the same in both codes.
- **yellow / teal lines**: rounds in the rep code (d = 15) and in LDPC (d = 16).

**Rounds above the dotted line: the Toffoli count overstates the saving.** The gap is largest for small tables, where the Swap is a large part of the lookup, and vanishes for large ones. The table below gives the absolute numbers and rounds per Toffoli.

In [ ]:
from matplotlib.ticker import FixedLocator, NullLocator, PercentFormatter

# colour follows the code in the lookup and prep figures, as the chart
# palette's series (figure_style: the rep code S1 yellow, LDPC S2 teal); the
# Toffoli count is the same in both codes, so it is one reference line, in
# S5 near-black, dotted
CODE_SERIES = {'rep code': 0, 'LDPC': 1}
CODE_COLOR = {code: fs.SERIES[i] for code, i in CODE_SERIES.items()}
CODE_EDGE = {code: fs.OUTLINE if i == 0 else fs.SERIES[i] for code, i in CODE_SERIES.items()}
CODE_MARKER = {'rep code': 'o', 'LDPC': 's'}
TOFFOLI_STYLE = dict(color=fs.S5, linewidth=1.2, ls=':')


def relative_panel(ax, ks, toffoli, rounds, title, note=None):
    '''
    One panel: cost relative to the plain QROM (K = 1) against K.
    toffoli:  {K: % of QROM}, drawn as the dotted reference line
    rounds:   {code: ({K: % of QROM}, {K: viable})}, one line per code;
              hollow markers are not viable
    note:     optional lines under the title, in muted ink
    '''
    style_axes(ax)
    ax.axhline(100, color=INK['axis'], linewidth=1, zorder=1)
    ax.plot(ks, [toffoli[K] for K in ks], zorder=2, **TOFFOLI_STYLE)
    for code, (pct, ok) in rounds.items():
        ax.plot(ks, [pct[K] for K in ks], zorder=3, **fs.styles(CODE_SERIES[code], lw=1.5)[0])
        for K in ks:
            ax.scatter(K, pct[K], s=45, marker=CODE_MARKER[code], zorder=4,
                       linewidth=0.9 if CODE_SERIES[code] == 0 else 1.5,
                       facecolor=CODE_COLOR[code] if ok[K] else INK['surface'],
                       edgecolor=CODE_EDGE[code])
    ax.set_xscale('log', base=2)
    ax.xaxis.set_major_locator(FixedLocator(ks))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xticklabels(['1\nQROM'] + [str(K) for K in ks[1:]])
    ax.yaxis.set_major_formatter(PercentFormatter(decimals=0))
    ax.margins(x=0.08, y=0.12)
    ax.set_xlabel('swap parameter K', color=INK['secondary'])
    lines = note.count('\n') + 1 if note else 0
    ax.set_title(title, color=INK['primary'], fontsize=10.5, loc='left', pad=8 + 11*lines)
    if note:
        ax.text(0, 1.015, note, transform=ax.transAxes, fontsize=8.5, color=INK['muted'],
                va='bottom', linespacing=1.3)


def relative_legend(fig, codes, toffoli_label, hollow=None):
    '''one legend row under the figure title'''
    handles, labels = [Line2D([], [], **TOFFOLI_STYLE)], [toffoli_label]
    handles += [fs.handle(CODE_SERIES[c], marker=CODE_MARKER[c], lw=1.5, ms=6) for c in codes]
    labels += [f"rounds, {c} ({distance_text(CODES[c]['distances'])})" for c in codes]
    if hollow:
        handles.append(Line2D([], [], linestyle='', marker='o', markersize=6,
                              markerfacecolor=INK['surface'], markeredgecolor=INK['secondary'],
                              markeredgewidth=1.5))
        labels.append(hollow)
    fig.legend(handles, labels, loc='upper left', ncol=len(handles), frameon=False, fontsize=9,
               labelcolor=INK['secondary'], bbox_to_anchor=(0.005, 0.955))

In [ ]:
# one clean SelectSwap lookup on its own, against a plain QROM over the same
# addresses: Toffoli count and rounds as a percentage of the plain QROM's.
# No budget -- just the lookup primitives of each cost model
lookup_addresses = (64, 512, 4096)   # table sizes to compare
lookup_bits = 9                      # output register width
lookup_alphas = range(0, 5)          # K = 1 .. 16

lookup_results = {}                  # (code, address, K) -> accounting.summary
for code, c in CODES.items():
    for address in lookup_addresses:
        for a in lookup_alphas:
            lookup_results[(code, address, 2**a)] = acc.summary(
                c['model'].SelectSwap_cost(address, lookup_bits, a, **c['distances'], variety='clean'))

ks = [2**a for a in lookup_alphas]
fig, axes = plt.subplots(1, len(lookup_addresses), figsize=(12, 4.6))
fig.patch.set_facecolor(INK['surface'])
for ax, address in zip(axes, lookup_addresses):
    pct = lambda code, field: {K: 100*lookup_results[(code, address, K)][field]
                                  / lookup_results[(code, address, 1)][field] for K in ks}
    rounds = {code: (pct(code, 'Temporal cost'), {K: True for K in ks}) for code in CODES}
    relative_panel(ax, ks, pct('rep code', 'Toffoli cost'), rounds,
                   f'{address:,} addresses')
axes[0].set_ylabel('% of the plain QROM (lower is cheaper)', color=INK['secondary'])
relative_legend(fig, CODES, 'Toffoli count (the same in both codes)')
fig.suptitle(f'One clean SelectSwap lookup ({lookup_bits}-bit output) relative to a plain QROM',
             color=INK['primary'], fontsize=12, x=0.01, ha='left')
fig.tight_layout(rect=(0, 0, 1, 0.9))
plt.show()

In [ ]:
# table view of the lookup figure
print(f"{'code':<10}{'addresses':>10}{'K':>4}{'Toffoli':>9}{'% QROM':>8}{'rounds':>9}{'% QROM':>8}"
      f"{'rounds/Toffoli':>16}")
for (code, address, K_row), s in lookup_results.items():
    q = lookup_results[(code, address, 1)]
    print(f"{code:<10}{address:>10,}{K_row:>4}{s['Toffoli cost']:>9}"
          f"{100*s['Toffoli cost']/q['Toffoli cost']:>7.0f}%{s['Temporal cost']:>9}"
          f"{100*s['Temporal cost']/q['Temporal cost']:>7.0f}%"
          f"{s['Temporal cost']/s['Toffoli cost']:>16.1f}")

## Prep lookups: plain QROM against SelectSwap

The same comparison for the lookups inside OuterPrep and InnerPrep, for the chosen parameter set (`system`).

**What is plotted.** Each prep is built by `optimize.prep_options` with the same K forced at every level: OuterPrep has levels of 1, 2, 4, …, 256 addresses, InnerPrep of RC·1, RC·2, …, RC·16 (a level with fewer than K addresses uses the largest K it can). From each prep, only its **lookups** are plotted: the prep record's `'lookups'`, the sum over its levels of that level's SelectSwap (Select, Swap, and for clean measuring out the unselected copies; for dirty the doubled Select and Swap and the Hadamard conjugation). Their Toffoli count and rounds are divided by the same prep's lookups when every level is a plain QROM (K = 1, 100 %; lower is cheaper).

- **Top row**: clean SelectSwap, in both codes.
- **Bottom row**: dirty SelectSwap, LDPC only (the rep code has no Z access to the system qubits it would borrow).
- **Hollow markers**: the prep option does not fit the parameter set's compute-zone budget (or, for dirty, borrows more than the 2N system qubits).

**Not plotted: the rest of the prep** (adders, rotations, shuffles, MBUs, reaction waits and the phase fixup). It costs exactly the same at every K, so it says nothing about QROM against SelectSwap, and its numbers are given under each panel title. It is left out because it would distort the comparison: it has few Toffolis but many rounds (e.g. OuterPrep, rep code: 103 Toffolis but 5,291 rounds, against 466 Toffolis and 12,129 rounds for the QROM lookups), so adding the same amount to every option pulls the rounds percentage towards 100 % far more than the Toffoli percentage, which makes a gap appear even where the lookups track the Toffoli count exactly. The whole-prep numbers are in the table below the figure.

In [ ]:
# every prep SelectSwap option, costed as the whole prep (optimize.prep_options):
# a plain QROM, and each variety at K = 2**alpha forced at every level.  The
# rep code only has QROM and clean SelectSwap (no Z access to the system
# qubits); LDPC also has the dirty variety
prep_bits = 9                    # the preps' bit precision, b_coef
prep_alphas = range(0, 5)        # K = 1 .. 16
VARIETIES_PER_CODE = {'rep code': ('clean',), 'LDPC': ('clean', 'dirty')}


def prep_zones(prep):
    '''the prep's positional arguments and required zones, for the chosen system'''
    p = SYSTEMS[system]
    n_r = int(np.ceil(np.log2(p['R'])))
    X, RC = p['N'] + 2**n_r*p['C'], 2**n_r*p['C']
    if prep == 'OuterPrep':
        return (X, prep_bits), dict(N=p['N'], b_rot=p['b_rot'], B=p['B'])
    return (RC, p['B'], prep_bits), dict(N=p['N'], b_rot=p['b_rot'], X=X)


PREPS = ('OuterPrep', 'InnerPrep')
prep_results = {(prep, code): opt.prep_options(getattr(hl, prep), prep_zones(prep)[0],
                                               SYSTEMS[system]['budget'][code], CODES[code]['distances'],
                                               model=CODES[code]['model'], alphas=prep_alphas,
                                               varieties=VARIETIES_PER_CODE[code],
                                               **prep_zones(prep)[1])
                for prep in PREPS for code in CODES}

In [ ]:
# the preps' lookups (prep_results above), a row per SelectSwap variety: the
# Toffoli count and rounds of each prep's SelectSwap lookups alone (its
# 'lookups' record), relative to the same prep's lookups as plain QROMs.  The
# rest of the prep costs the same at every K, so it is left out and stated
# under each panel title.  The dirty variety only exists in LDPC
prep_ks = [2**a for a in prep_alphas]
ROW_VARIETIES = (('clean', ('rep code', 'LDPC')), ('dirty', ('LDPC',)))


def lookup_totals(r):
    '''(Toffoli, rounds) of a prep record's lookups alone'''
    return acc.total(r['lookups'], 'toffoli'), acc.total(r['lookups'], 'rounds')


def rest_totals(r):
    '''(Toffoli, rounds) of the rest of the prep: everything but its lookups'''
    s, (t, n) = acc.summary(r), lookup_totals(r)
    return s['Toffoli cost'] - t, s['Temporal cost'] - n


fig, axes = plt.subplots(len(ROW_VARIETIES), len(PREPS), figsize=(12, 9.4))
fig.patch.set_facecolor(INK['surface'])
toffoli_gap = 0
for i, (variety, codes) in enumerate(ROW_VARIETIES):
    for j, prep in enumerate(PREPS):
        def by_k(code):
            options = prep_results[(prep, code)]
            qrom = next(r for r in options if r['method'] == 'qrom')
            return qrom, {r['K']: r for r in options if r['method'] == variety} | {1: qrom}

        def pct(code, which):
            qrom, rows = by_k(code)
            return {K: 100*lookup_totals(rows[K])[which]/lookup_totals(qrom)[which]
                    for K in prep_ks}

        rounds = {code: (pct(code, 1), {K: by_k(code)[1][K]['viable'] for K in prep_ks})
                  for code in codes}
        toffoli = pct(codes[0], 0)
        # the Toffoli count is drawn once: how far the codes' counts differ
        for code in codes[1:]:
            other = pct(code, 0)
            toffoli_gap = max(toffoli_gap, max(abs(other[K] - toffoli[K]) for K in prep_ks))
        # the rest of the prep, one line per code
        rest = '\n'.join(f"  {code}: {t:,} Toffolis, {n:,} rounds"
                         for code in codes for t, n in [rest_totals(by_k(code)[0])])
        relative_panel(axes[i, j], prep_ks, toffoli, rounds,
                       f'{prep} lookups, {variety} SelectSwap'
                       + ('  (LDPC only)' if len(codes) == 1 else ''),
                       note=f'rest of the prep, the same at every K (not plotted):\n{rest}')
    axes[i, 0].set_ylabel('% of the plain-QROM lookups\n(lower is cheaper)',
                          color=INK['secondary'])
toffoli_label = ('Toffoli count (the same in both codes)' if toffoli_gap < 0.05
                 else f'Toffoli count (codes within {toffoli_gap:.1f} points)')
relative_legend(fig, CODES, toffoli_label, hollow='over the compute-zone budget')
budgets = ', '.join(f"{code} ≤ {SYSTEMS[system]['budget'][code]}" for code in CODES)
fig.suptitle(f"The preps' lookups relative to plain QROMs, parameter set {system} "
             f'(b_coef = {prep_bits}; compute zone {budgets})',
             color=INK['primary'], fontsize=12, x=0.01, ha='left')
fig.tight_layout(rect=(0, 0, 1, 0.93))
plt.show()

In [ ]:
# table view of the prep figure: every option with its Toffoli cost, rounds,
# logical cycles, rounds per Toffoli and compute zone, and why it is not viable
for prep in PREPS:
    for code in CODES:
        print(f"{prep}, {code}, {distance_text(CODES[code]['distances'])}, "
              f"compute zone <= {SYSTEMS[system]['budget'][code]}")
        opt.show_prep_options(prep_results[(prep, code)], model=CODES[code]['model'],
                              distances=CODES[code]['distances'])
        print()

### Which lookups the block encoding uses

For the actual parameter sets: at each level of each prep call in the block encoding, is the lookup a plain QROM or a SelectSwap, and with which K? This is the choice `BE_cost` makes, not a forced one: the SelectSwap variety set in `SETTINGS_54` / `SETTINGS_76` (`outer_variety`, `inner_variety`), and K picked per level for the fewest rounds (`SelectSwap_opt`), up to the largest K the variety allows: for clean, whose copies and access corridors fit the compute zone (`alpha_budget`); for dirty, whose borrowed copies fit the 2N system qubits. Each column is one level, labelled by its number of addresses.

In [ ]:
# the lookup the block encoding actually uses at each level of each prep call,
# for both parameter sets and codes: K = 1 is a plain QROM, K > 1 a SelectSwap
# of the variety set in SETTINGS_54 / SETTINGS_76 (shown in brackets).  As in
# BE_cost: K picked per level by SelectSwap_opt (fewest rounds) up to the
# largest K the variety allows (clean: the compute zone, alpha_budget; dirty:
# the 2N borrowable system qubits), at each parameter set's own distances
# at NOISE_LEVEL (code_distances)
BE_PREP_CALLS = (('OuterPrep', False, 1), ('OuterPrep', True, 1),
                 ('InnerPrep', False, 2), ('InnerPrep', True, 2))   # (prep, adjoint, calls)



def prep_variety(sys_key, code, prep):
    '''the prep's SelectSwap variety, as set in the system's SETTINGS'''
    key = 'outer_variety' if prep == 'OuterPrep' else 'inner_variety'
    return SETTINGS_BY_SYSTEM[sys_key][CODES[code]['model']]['option'][key]


def be_prep_levels(sys_key, code, prep, adjoint):
    '''[(addresses, K)] per level, as BE_cost builds the prep'''
    p, c = SYSTEMS[sys_key], CODES[code]
    distances = code_distances(sys_key, c['model'], NOISE_LEVEL)
    n_r = int(np.ceil(np.log2(p['R'])))
    X, RC = p['N'] + 2**n_r*p['C'], 2**n_r*p['C']
    common = dict(adjoint=adjoint, model=c['model'], N=p['N'], b_rot=p['b_rot'],
                  buffer_budget=layout.buffer_zone_size(),
                  variety=prep_variety(sys_key, code, prep))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')        # budget warnings are not the point here
        if prep == 'OuterPrep':
            r = hl.OuterPrep(X, prep_bits, p['budget'][code], distances, B=p['B'], **common)
        else:
            r = hl.InnerPrep(RC, p['B'], prep_bits, p['budget'][code], distances, X=X, **common)
    return r['levels']


for sys_key, p in SYSTEMS.items():
    print(f"Parameter set {sys_key}: N = {p['N']}, B = {p['B']}, R = {p['R']}, C = {p['C']}, "
          f"b_coef = {prep_bits}")
    for code, c in CODES.items():
        K_max = 2**hl.alpha_budget(p['budget'][code], prep_bits, model=c['model'])
        dirty = [prep for prep in ('OuterPrep', 'InnerPrep')
                 if prep_variety(sys_key, code, prep) == 'dirty']
        K_dirty = 2**int(np.floor(np.log2(1 + 2*p['N']/prep_bits)))
        print(f"  {code}: {distance_text(code_distances(sys_key, c['model'], NOISE_LEVEL))}, compute zone <= {p['budget'][code]}, "
              f"so clean K <= {K_max}"
              + (f"; dirty ({', '.join(dirty)}) borrows <= 2N = {2*p['N']} system "
                 f"qubits, so K <= {K_dirty}" if dirty else ''))
    for prep in ('OuterPrep', 'InnerPrep'):
        calls = [(adj, n) for name, adj, n in BE_PREP_CALLS if name == prep]
        levels = {(code, adj): be_prep_levels(sys_key, code, prep, adj)
                  for code in CODES for adj, _ in calls}
        # forward and adjoint pick the same K: one row per code then
        same = all(levels[(code, False)] == levels[(code, True)] for code in CODES)
        label = ' and '.join(f"{prep}{'_dag' if adj else ''} ({n} per block encoding)"
                             for adj, n in calls)
        print(f"\n  {label}")
        sizes = [L for L, _ in levels[(next(iter(CODES)), False)]]
        print(f"    {'addresses':<16}" + ''.join(f'{L:>8,}' for L in sizes))
        for (code, adj), lv in levels.items():
            if same and adj:
                continue
            name = f"{code} ({prep_variety(sys_key, code, prep)})" + ('' if same else f"{', dag' if adj else ''}")
            print(f"    {name:<16}" + ''.join(f"{'QROM' if K == 1 else f'K={K}':>8}" for _, K in lv))
    print('\n')